#blas.dgemm

In [1]:
import numpy as np
import time
import csv

from scipy.linalg import blas

def benchmark_matrix_multiplication(matrix_size):
    A = np.random.rand(matrix_size, matrix_size)
    B = np.random.rand(matrix_size, matrix_size)

    start_time = time.time()
    C = blas.dgemm(alpha=1.0, a=A, b=B)
    end_time = time.time()

    execution_time = end_time - start_time

    return matrix_size, execution_time

matrix_sizes = range(500,6000,100)

# Mesurer le temps d'exécution pour chaque taille de matrice
with open('resultats_cblas.csv', 'w', newline='') as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerow(['Taille de la matrice', 'Temps d\'exécution (s)'])

    for size in matrix_sizes:
        matrix_size, execution_time = benchmark_matrix_multiplication(size)
        csv_writer.writerow([matrix_size, execution_time])

print("Résultats enregistrés dans le fichier 'resultats_cblas.csv'")


KeyboardInterrupt: 

#pragma omp parallel section

In [2]:
%%writefile codeomparallelsection.cpp
#include <iostream>
#include <fstream>
#include <vector>
#include <cstdlib>
#include <ctime>
#include <omp.h>

#include <cblas.h>

void benchmark_matrix_multiplication(int matrix_size, std::vector<std::pair<int, double>>& results) {
    double* A = (double*)malloc(matrix_size * matrix_size * sizeof(double));
    double* B = (double*)malloc(matrix_size * matrix_size * sizeof(double));
    double* C = (double*)malloc(matrix_size * matrix_size * sizeof(double));

    #pragma omp parallel for
    for (int i = 0; i < matrix_size * matrix_size; ++i) {
        A[i] = static_cast<double>(rand()) / RAND_MAX;
        B[i] = static_cast<double>(rand()) / RAND_MAX;
    }

    double start_time = omp_get_wtime();

    cblas_dgemm(CblasRowMajor, CblasNoTrans, CblasNoTrans, matrix_size, matrix_size, matrix_size, 1.0, A, matrix_size, B, matrix_size, 0.0, C, matrix_size);

    double end_time = omp_get_wtime();
    double execution_time = end_time - start_time;

    free(A);
    free(B);
    free(C);

    #pragma omp critical
    results.push_back(std::make_pair(matrix_size, execution_time));
}

int main() {
    int matrix_sizes[] = {500, 600, 700, 800, 900, 1000, 1100,1200,1300,1400,1500,1600,1700,1800,1900,2000,2100, 2200,2300,2400,2500,2600,2700,2800,2900,3000,3100,3200,3300};

    int num_sizes = sizeof(matrix_sizes) / sizeof(matrix_sizes[0]);


    std::vector<std::pair<int, double>> results;

    #pragma omp parallel sections
    {
        #pragma omp section
        {
            for (int i = 0; i < num_sizes; ++i) {
                benchmark_matrix_multiplication(matrix_sizes[i], results);
            }
        }
    }

    std::ofstream csvfile("resultats_parallelsection_omp.csv");
    csvfile << "Taille de la matrice, Temps d'exécution (s)\n";

    for (const auto& result : results) {
        csvfile << result.first << "," << result.second << "\n";
    }

    std::cout << "Résultats enregistrés dans le fichier 'resultats_parallelsection_omp.csv'" << std::endl;

    return 0;
}


Writing codeomparallelsection.cpp


In [3]:
!g++ -fopenmp codeomparallelsection.cpp -o codeomparallelsection -lblas
!./codeomparallelsection

Résultats enregistrés dans le fichier 'resultats_parallelsection_omp.csv'


#pragma omp parallel for

In [ ]:
%%writefile codeomparallelfor.cpp
#include <iostream>
#include <fstream>
#include <vector>
#include <cstdlib>
#include <ctime>
#include <omp.h>
#include <numeric>


void complex_computation(int matrix_size, std::vector<std::pair<int, double>>& results) {
    double result = 0.0;
    for (int i = 0; i < matrix_size * matrix_size; ++i) {
        result += static_cast<double>(rand()) / RAND_MAX;
    }

    double start_time = omp_get_wtime();
    #pragma omp parallel for
    for (int i = 0; i < matrix_size * matrix_size; ++i) {
        result = result * 2.0 + i;
    }
    double end_time = omp_get_wtime();
    double execution_time = end_time - start_time;

    #pragma omp critical
    results.push_back(std::make_pair(matrix_size, execution_time));
}

int main() {
    int matrix_sizes[] = {500, 600, 700, 800, 900, 1000, 1100,1200,1300,1400,1500,1600,1700,1800,1900,2000,2100, 2200,2300,2400,2500,2600,2700,2800,2900,3000,3100,3200,3300};


    int num_sizes = sizeof(matrix_sizes) / sizeof(matrix_sizes[0]);

    std::vector<std::pair<int, double>> results;

    #pragma omp parallel for
    for (int i = 0; i < num_sizes; ++i) {
        complex_computation(matrix_sizes[i], results);
    }

    std::ofstream csvfile("resultats_omparallel_for.csv");
    csvfile << "Taille de la matrice, Temps d'exécution (s)\n";

    for (const auto& result : results) {
        csvfile << result.first << "," << result.second << "\n";
    }

    std::cout << "Résultats enregistrés dans le fichier 'resultats_omparallel_for.csv'" << std::endl;

    return 0;
}


Writing codeomparallelfor.cpp


In [ ]:
%%time
!g++ codeomparallelfor.cpp -fopenmp -o codeomparallelfor
!./codeomparallelfor

Résultats enregistrés dans le fichier 'resultats_omparallel_for.csv'
CPU times: user 54.2 ms, sys: 11.4 ms, total: 65.6 ms
Wall time: 6.77 s


#pragma omp section




In [ ]:
%%writefile codeompsection.cpp
#include <iostream>
#include <fstream>
#include <vector>
#include <cstdlib>
#include <ctime>
#include <omp.h>

void complex_computation(int matrix_size, std::vector<std::pair<int, double>>& results) {
    double result = 0.0;
    for (int i = 0; i < matrix_size * matrix_size; ++i) {
        result += static_cast<double>(rand()) / RAND_MAX;
    }

    double start_time = omp_get_wtime();

    #pragma omp sections
    {
        #pragma omp section
        {
            // First section of computation
            for (int i = 0; i < matrix_size * matrix_size / 2; ++i) {
                result = result * 2.0 + i;
            }
        }

        #pragma omp section
        {
            // Second section of computation
            for (int i = matrix_size * matrix_size / 2; i < matrix_size * matrix_size; ++i) {
                result = result * 2.0 + i;
            }
        }
    }

    double end_time = omp_get_wtime();
    double execution_time = end_time - start_time;

    #pragma omp critical
    results.push_back(std::make_pair(matrix_size, execution_time));
}

int main() {
    int matrix_sizes[] = {500, 600, 700, 800, 900, 1000, 1100,1200,1300,1400,1500,1600,1700,1800,1900,2000,2100,2200,2300,2400,2500,2600,2700,2800,2900,3000};

    int num_sizes = sizeof(matrix_sizes) / sizeof(matrix_sizes[0]);

    std::vector<std::pair<int, double>> results;

    #pragma omp parallel for
    for (int i = 0; i < num_sizes; ++i) {
        complex_computation(matrix_sizes[i], results);
    }

    std::ofstream csvfile("resultats_omp_sections.csv");
    csvfile << "Taille de la matrice, Temps d'exécution (s)\n";

    for (const auto& result : results) {
        csvfile << result.first << "," << result.second << "\n";
    }

    std::cout << "Résultats enregistrés dans le fichier 'resultats_omp_sections.csv'" << std::endl;

    return 0;
}


Writing codeompsection.cpp


In [ ]:
!g++ codeompsection.cpp -fopenmp -o codeompsection
!./codeompsection

Résultats enregistrés dans le fichier 'resultats_omp_sections.csv'


# pragma omp for

In [ ]:
%%writefile codeompfor.cpp
#include <iostream>
#include <fstream>
#include <vector>
#include <cstdlib>
#include <ctime>
#include <omp.h>

void complex_computation(int matrix_size, std::vector<std::pair<int, double>>& results) {
    double result = 0.0;
    for (int i = 0; i < matrix_size * matrix_size; ++i) {
        result += static_cast<double>(rand()) / RAND_MAX;
    }

    double start_time = omp_get_wtime();

    #pragma omp for
    for (int i = 0; i < matrix_size * matrix_size; ++i) {
        result = result * 2.0 + i;
    }

    double end_time = omp_get_wtime();
    double execution_time = end_time - start_time;

    #pragma omp critical
    results.push_back(std::make_pair(matrix_size, execution_time));
}

int main() {
    int matrix_sizes[] = {500, 600, 700, 800, 900, 1000, 1100,1200,1300,1400,1500,1600,1700,1800,1900,2000,2100,2200,2300,2400,2500,2600,2700,2800,2900,3000};

    int num_sizes = sizeof(matrix_sizes) / sizeof(matrix_sizes[0]);

    std::vector<std::pair<int, double>> results;

    #pragma omp parallel
    {
        #pragma omp for
        for (int i = 0; i < num_sizes; ++i) {
            complex_computation(matrix_sizes[i], results);
        }
    }

    std::ofstream csvfile("resultats_omp_for.csv");
    csvfile << "Taille de la matrice, Temps d'exécution (s)\n";

    for (const auto& result : results) {
        csvfile << result.first << "," << result.second << "\n";
    }

    std::cout << "Résultats enregistrés dans le fichier 'resultats_omp_for.csv'" << std::endl;

    return 0;
}


Writing codeompfor.cpp


In [ ]:
!g++ codeompfor.cpp -fopenmp -o codeompfor
!./codeompfor

Résultats enregistrés dans le fichier 'resultats_omp_for.csv'


# Regression Lineaire

In [ ]:
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression



# Lire le fichier CSV avec pandas
data = pd.read_csv("resultats_omp_sections.csv")


# Supprimer les colonnes non nécessaires du DataFrame si nécessaire
# Par exemple, si la première colonne est l'index, vous pouvez la supprimer.
#data = data.drop('', axis=1)

# Diviser les données en variables d'entrée (X) et de sortie (y)
# Toutes les colonnes sauf la dernière
# La dernière colonne

X = data.iloc[:, :-1]
y = data.iloc[:, -1]

# Créer un modèle de régression linéaire polynomiale
poly = PolynomialFeatures(degree=3)
X_poly = poly.fit_transform(X)
X_poly=X_poly[:,1:]
print(X_poly)
model = LinearRegression()

# Entraîner le modèle sur les données d'entrée transformées
model.fit(X_poly, y)

# Imprimer les coefficients du modèle (les coefficients du polynôme)
coefficients = model.coef_
intercept = model.intercept_

print("Coefficients du polynôme:", coefficients)
print("Terme indépendant:", intercept)
print(model.score(X_poly, y))


[[1.8000e+03 3.2400e+06 5.8320e+09]
 [5.0000e+02 2.5000e+05 1.2500e+08]
 [1.9000e+03 3.6100e+06 6.8590e+09]
 [6.0000e+02 3.6000e+05 2.1600e+08]
 [7.0000e+02 4.9000e+05 3.4300e+08]
 [2.0000e+03 4.0000e+06 8.0000e+09]
 [8.0000e+02 6.4000e+05 5.1200e+08]
 [2.1000e+03 4.4100e+06 9.2610e+09]
 [9.0000e+02 8.1000e+05 7.2900e+08]
 [2.2000e+03 4.8400e+06 1.0648e+10]
 [1.0000e+03 1.0000e+06 1.0000e+09]
 [2.3000e+03 5.2900e+06 1.2167e+10]
 [2.4000e+03 5.7600e+06 1.3824e+10]
 [1.1000e+03 1.2100e+06 1.3310e+09]
 [1.2000e+03 1.4400e+06 1.7280e+09]
 [2.5000e+03 6.2500e+06 1.5625e+10]
 [1.3000e+03 1.6900e+06 2.1970e+09]
 [2.6000e+03 6.7600e+06 1.7576e+10]
 [2.7000e+03 7.2900e+06 1.9683e+10]
 [1.4000e+03 1.9600e+06 2.7440e+09]
 [2.8000e+03 7.8400e+06 2.1952e+10]
 [1.5000e+03 2.2500e+06 3.3750e+09]
 [2.9000e+03 8.4100e+06 2.4389e+10]
 [1.6000e+03 2.5600e+06 4.0960e+09]
 [1.7000e+03 2.8900e+06 4.9130e+09]
 [3.0000e+03 9.0000e+06 2.7000e+10]]
Coefficients du polynôme: [ 1.31513308e-03 -8.76968957e-07  1.6

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

data = pd.read_csv("resultats_cblas.csv")

X = data.iloc[:, :-1]
y = data.iloc[:, -1]

poly = PolynomialFeatures(degree=3)
X_poly = poly.fit_transform(X)
X_poly = X_poly[:, 1:]

# Entraîner le modèle sur les données d'entrée transformées
model = LinearRegression()
model.fit(X_poly, y)

# Générer des prédictions pour les données d'entraînement
y_pred = model.predict(X_poly)

# Créer un DataFrame avec les données réelles et les prédictions
df = pd.DataFrame({'X': X.iloc[:, 0], 'y': y, 'Predictions': y_pred})

# Trier le DataFrame par la variable d'entrée pour un tracé plus lisse
df = df.sort_values(by='X')

# Créer un graphique à dispersion avec la ligne de régression polynomiale
fig = px.scatter(df, x='X', y='y', title='Régression Polynomiale cblas')
fig.add_trace(px.line(df, x='X', y='Predictions').data[0])

fig.show()


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

data = pd.read_csv("/content/resultats_omparallel_for.csv")

X = data.iloc[:, :-1]
y = data.iloc[:, -1]

poly = PolynomialFeatures(degree=3)
X_poly = poly.fit_transform(X)
X_poly = X_poly[:, 1:]

# Entraîner le modèle sur les données d'entrée transformées
model = LinearRegression()
model.fit(X_poly, y)

# Générer des prédictions pour les données d'entraînement
y_pred = model.predict(X_poly)

# Créer un DataFrame avec les données réelles et les prédictions
df = pd.DataFrame({'X': X.iloc[:, 0], 'y': y, 'Predictions': y_pred})

# Trier le DataFrame par la variable d'entrée pour un tracé plus lisse
df = df.sort_values(by='X')

# Créer un graphique à dispersion avec la ligne de régression polynomiale
fig = px.scatter(df, x='X', y='y', title='Régression Polynomiale ( parallel for)')
fig.add_trace(px.line(df, x='X', y='Predictions').data[0])

fig.show()


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

data = pd.read_csv("/content/resultats_parallelsection_omp.csv")

X = data.iloc[:, :-1]
y = data.iloc[:, -1]

poly = PolynomialFeatures(degree=3)
X_poly = poly.fit_transform(X)
X_poly = X_poly[:, 1:]

# Entraîner le modèle sur les données d'entrée transformées
model = LinearRegression()
model.fit(X_poly, y)

# Générer des prédictions pour les données d'entraînement
y_pred = model.predict(X_poly)

# Créer un DataFrame avec les données réelles et les prédictions
df = pd.DataFrame({'X': X.iloc[:, 0], 'y': y, 'Predictions': y_pred})

# Trier le DataFrame par la variable d'entrée pour un tracé plus lisse
df = df.sort_values(by='X')

# Créer un graphique à dispersion avec la ligne de régression polynomiale
fig = px.scatter(df, x='X', y='y', title='Régression Polynomiale (parallel section)')
fig.add_trace(px.line(df, x='X', y='Predictions').data[0])

fig.show()


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

data = pd.read_csv("/content/resultats_omp_sections.csv")

X = data.iloc[:, :-1]
y = data.iloc[:, -1]

poly = PolynomialFeatures(degree=3)
X_poly = poly.fit_transform(X)
X_poly = X_poly[:, 1:]

# Entraîner le modèle sur les données d'entrée transformées
model = LinearRegression()
model.fit(X_poly, y)

# Générer des prédictions pour les données d'entraînement
y_pred = model.predict(X_poly)

# Créer un DataFrame avec les données réelles et les prédictions
df = pd.DataFrame({'X': X.iloc[:, 0], 'y': y, 'Predictions': y_pred})

# Trier le DataFrame par la variable d'entrée pour un tracé plus lisse
df = df.sort_values(by='X')

# Créer un graphique à dispersion avec la ligne de régression polynomiale
fig = px.scatter(df, x='X', y='y', title='Régression Polynomiale(section)')
fig.add_trace(px.line(df, x='X', y='Predictions').data[0])

fig.show()
